In [1]:
import json

file_path = "data/beacon-traffic-sample.jsonl"

In [2]:
with open(file_path, "r") as f:
    first_line = f.readline()

print(first_line)

{"request_id": "req_00000", "tenant_id": "t154", "timestamp": "2026-07-14T00:00:17.528Z", "conversation_id": "conv_00001", "endpoint": "analysis", "system_prompt_tokens": 667, "input_tokens": 11791, "shared_prefix_tokens": 667, "output_tokens": 782}



In [3]:
import pandas as pd

df = pd.read_json(file_path, lines=True)

df.shape


(5000, 9)

In [4]:
df.head()

,request_id,tenant_id,timestamp,conversation_id,endpoint,system_prompt_tokens,input_tokens,shared_prefix_tokens,output_tokens
0,req_00000,t154,2026-07-14 00:00:17.528000+00:00,conv_00001,analysis,667,11791,667,782
1,req_00001,t181,2026-07-14 00:00:44.485000+00:00,conv_00002,chat,2060,2126,2060,455
2,req_00002,t181,2026-07-14 00:01:17.501000+00:00,conv_00002,chat,2060,2772,2581,143
3,req_00003,t181,2026-07-14 00:01:21.404000+00:00,conv_00002,chat,2060,3008,2915,216
4,req_00004,t164,2026-07-14 00:01:22.508000+00:00,conv_00003,chat,6135,6174,6135,252


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype              
---  ------                --------------  -----              
 0   request_id            5000 non-null   str                
 1   tenant_id             5000 non-null   str                
 2   timestamp             5000 non-null   datetime64[us, UTC]
 3   conversation_id       5000 non-null   str                
 4   endpoint              5000 non-null   str                
 5   system_prompt_tokens  5000 non-null   int64              
 6   input_tokens          5000 non-null   int64              
 7   shared_prefix_tokens  5000 non-null   int64              
 8   output_tokens         5000 non-null   int64              
dtypes: datetime64[us, UTC](1), int64(4), str(4)
memory usage: 351.7 KB


In [6]:
df.describe()

,system_prompt_tokens,input_tokens,shared_prefix_tokens,output_tokens
count,5000.000000,5000.000000,5000.000000,5000.000000
mean,3498.414600,6395.063600,4568.296200,255.987200
std,2470.129136,5309.968754,2859.344429,172.206645
min,330.000000,2093.000000,330.000000,25.000000
25%,2060.000000,3253.000000,2691.750000,146.000000
50%,2451.000000,4656.000000,3950.500000,211.000000
75%,4195.000000,7110.750000,5988.000000,314.000000
max,16000.000000,32000.000000,21394.000000,2136.000000


In [7]:
df["endpoint"].value_counts()

endpoint
chat        4550
analysis     450
Name: count, dtype: int64

In [8]:
df["conversation_id"].nunique()

1430

In [9]:
df["prefix_fraction"] = df["shared_prefix_tokens"] / df["input_tokens"]
df["prefix_fraction"].describe()

count    5000.000000
mean        0.886278
std         0.267840
min         0.010971
25%         0.955677
50%         0.972814
75%         0.983087
max         0.997518
Name: prefix_fraction, dtype: float64

In [10]:
df.groupby("endpoint")[[
    "input_tokens",
    "shared_prefix_tokens",
    "output_tokens",
    "prefix_fraction"
]].median()

,input_tokens,shared_prefix_tokens,output_tokens,prefix_fraction
endpoint,,,,
analysis,18712.0,709.0,371.5,0.032901
chat,4334.0,4210.5,203.0,0.974946


In [11]:
df.groupby("conversation_id").size().describe()

count    1430.000000
mean        3.496503
std         3.445820
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max        14.000000
dtype: float64

- 5,000 requests across 1,430 conversations.
- Chat is 91% of requests.
- Median chat request: 4,334 input / 203 output tokens.
- Median chat shared-prefix fraction: 97.5%.
- Analysis requests are much larger and have little prefix reuse.

## Benchmark setup

- Provider: RunPod
- GPU: 1x NVIDIA H100 80GB HBM3
- Model: Qwen/Qwen3-8B
- Serving stack: vLLM
- Max context: 32,768 tokens

## Benchmark server

- GPU: NVIDIA H100 80GB HBM3
- Model: Qwen/Qwen3-8B
- Max context: 32,768
- Server reachable at localhost:8000 (haha local host funny joke)
- vLLM OpenAI-compatible endpoint verified successfully

### Chat cold-cache saturation test

- Input/output: 4,334 / 203 tokens
- Requests: 100
- Arrival rate: unlimited
- Request throughput: 5.55 req/s
- Total throughput: 25,188 tok/s
- Median TTFT: 5,825 ms
- p99 TTFT: 15,842 ms
- Median ITL: 26.3 ms
- p99 ITL: 435 ms
- Result: throughput saturation point; does not satisfy Beacon latency SLA

### Chat cold-cache @ 3 RPS

- Input/output: 4,334 / 203 tokens
- Configured arrival rate: 3 req/s
- Achieved request throughput: 2.83 req/s
- Total throughput: 12,860 tok/s
- p95 TTFT: 348 ms
- p95 ITL: 18.7 ms
- Result: PASS

### Chat cold-cache @ 4.5 RPS

- Achieved throughput: 4.05 req/s
- Total throughput: 18,386 tok/s
- p95 TTFT: 528 ms
- p95 ITL: 85.5 ms
- Result: FAIL — ITL exceeds 60 ms SLA

### Chat cold-cache @ 4.0 RPS
- Achieved throughput: 3.67 req/s
- Total throughput: 16,648 tok/s
- p95 TTFT: 427 ms
- p95 ITL: 79.7 ms
- Result: FAIL — ITL exceeds 60 ms

### Chat cold-cache @ 3.5 RPS

- Input/output: 4,334 / 203 tokens
- Configured arrival rate: 3.5 req/s
- Achieved request throughput: 3.26 req/s
- Total throughput: 14,803 tok/s
- p95 TTFT: 411 ms
- p95 ITL: 46.7 ms
- Result: PASS

### Chat cold-cache @ 3.75 RPS

- Input/output: 4,334 / 203 tokens
- Configured arrival rate: 3.75 req/s
- Achieved request throughput: 3.47 req/s
- Total throughput: 15,730 tok/s
- p95 TTFT: 457 ms
- p95 ITL: 71.1 ms
- Result: FAIL — ITL exceeds 60 ms SLA

### Chat cold-cache @ 3.6 RPS

- Input/output: 4,334 / 203 tokens
- Configured arrival rate: 3.6 req/s
- Achieved request throughput: 3.35 req/s
- Total throughput: 15,192 tok/s
- p95 TTFT: 405 ms
- p95 ITL: 43.1 ms
- Result: PASS

### Chat cold-cache @ 3.7 RPS

- Input/output: 4,334 / 203 tokens
- Configured arrival rate: 3.7 req/s
- Achieved request throughput: 3.43 req/s
- Total throughput: 15,559 tok/s
- p95 TTFT: 396 ms
- p95 ITL: 35.7 ms
- Result: PASS

Note: 3.75 RPS failed the ITL SLA while 3.7 passed, suggesting some run-to-run
variance from the Poisson arrival process. I treat ~3.5–3.7 RPS as the approximate
cold-cache SLA-safe region rather than claiming a precise cutoff.

### Warm-prefix saturation test — diagnostic

- Shared prefix / unique suffix: 4,210 / 124 tokens
- Intended output: 203 tokens
- Prefixes: 40
- Successful requests: 80
- Request throughput: 9.32 req/s
- Total reported token throughput: 41,915 tok/s
- p95 TTFT: 4,715 ms
- p95 ITL: 25.4 ms
- Result: FAIL — TTFT exceeds 2,000 ms SLA

Important: only 12,872 output tokens were generated (~161/request rather than
203) because requests could terminate at EOS. I therefore do not use this run
for direct cold-vs-warm comparison. Subsequent prefix tests use --ignore-eos.


### Chat warm-prefix saturation test

- Shared prefix / unique suffix: 4,210 / 124 tokens
- Total input: ~4,334 tokens/request
- Output: 203 tokens/request
- Prefixes: 40
- Requests: 80
- Arrival rate: unlimited
- Successful requests: 80
- Achieved request throughput: 14.84 req/s
- Reported total token throughput: 67,330 tok/s
- p95 TTFT: 1,063 ms
- p95 ITL: 26.7 ms
- Result: PASS

The warm-prefix workload remained within Beacon's latency SLA even when all
80 requests were submitted without rate limiting.

Caution: vLLM's reported "total token throughput" counts the full logical
input length, including cached prefix tokens. It should not be interpreted
as 67k tokens/s of fresh GPU prefill computation.

### Chat warm-prefix saturation test — 160 requests

- Shared prefix / unique suffix: 4,210 / 124 tokens
- Output: 203 tokens/request
- Prefixes: 40
- Requests: 160
- Arrival rate: unlimited
- Achieved request throughput: 12.85 req/s
- Reported total token throughput: 58,290 tok/s
- p95 TTFT: 4,962 ms
- p95 ITL: 46.6 ms
- Result: FAIL — TTFT exceeds 2,000 ms SLA

Interpretation: prefix reuse allows much higher throughput than the cold-cache case,
but sufficiently large bursts still create queueing and violate TTFT.

### Chat warm-prefix @ 10 RPS

- Shared prefix / unique suffix: 4,210 / 124 tokens
- Output: 203 tokens/request
- Prefixes: 40
- Configured arrival rate: 10 req/s
- Achieved request throughput: 8.90 req/s
- Reported total token throughput: 40,375 tok/s
- p95 TTFT: 106 ms
- p95 ITL: 15.9 ms
- Result: PASS

Interpretation: with strong prefix reuse, one H100 can handle far more
chat traffic than the cold-cache case while staying comfortably inside SLA.

### Chat warm-prefix @ 12 RPS

- Shared prefix / unique suffix: 4,210 / 124 tokens
- Total input: ~4,334 tokens/request
- Output: 203 tokens/request
- Prefixes: 40
- Configured arrival rate: 12 req/s
- Achieved request throughput: 10.28 req/s
- Reported total token throughput: 46,627 tok/s
- p95 TTFT: 79 ms
- p95 ITL: 17.3 ms
- Result: PASS

Interpretation: warm-prefix chat sustains dramatically more traffic than the
cold-cache workload while remaining well inside Beacon's SLA. I did not
continue searching for the exact warm-cache saturation point because the case
study's time budget favors establishing the effect rather than overfitting a
single-GPU boundary.

### Analysis saturation test

- Input/output: 18,712 / 372 tokens
- Requests: 40
- Arrival rate: unlimited
- Achieved request throughput: 0.92 req/s
- Total token throughput: 17,591 tok/s
- p95 TTFT: 32,896 ms
- p95 ITL: 265.7 ms
- Result: FAIL — both TTFT and ITL exceed Beacon SLA

Interpretation: analysis requests are much more prefill-heavy than chat and
cannot be served near the raw saturation point while meeting latency targets.

### Analysis @ 0.5 RPS

- Input/output: 18,712 / 372 tokens
- Configured arrival rate: 0.5 req/s
- Achieved request throughput: 0.48 req/s
- Total throughput: 9,131 tok/s
- p95 TTFT: 1,317 ms
- p95 ITL: 14.0 ms
- Result: PASS

### Analysis @ 0.7 RPS

- Input/output: 18,712 / 372 tokens
- Configured arrival rate: 0.7 req/s
- Achieved request throughput: 0.65 req/s
- Total throughput: 12,433 tok/s
- p95 TTFT: 1,367 ms
- p95 ITL: 21.3 ms
- Result: PASS

In [15]:
df["total_tokens"] = df["input_tokens"] + df["output_tokens"]

df.groupby("endpoint").agg(
    requests=("request_id", "count"),
    input_tokens=("input_tokens", "sum"),
    output_tokens=("output_tokens", "sum"),
    total_tokens=("total_tokens", "sum")
)


,requests,input_tokens,output_tokens,total_tokens
endpoint,,,,
analysis,450,8866631,201263,9067894
chat,4550,23108687,1078673,24187360


In [16]:
df.groupby("endpoint")["total_tokens"].sum() / df["total_tokens"].sum()

endpoint
analysis    0.272676
chat        0.727324
Name: total_tokens, dtype: float64

## Benchmark summary

### Chat, little/no prefix reuse
- SLA-safe region: approximately 3.5–3.7 offered req/s
- 3.7 RPS test:
  - achieved: 3.43 req/s
  - reported logical throughput: 15,559 tok/s
  - p95 TTFT: 396 ms
  - p95 ITL: 35.7 ms

### Chat, strong prefix reuse
- 4,210-token shared prefix + 124-token unique suffix
- 12 RPS test:
  - achieved: 10.28 req/s
  - reported logical throughput: 46,627 tok/s
  - p95 TTFT: 79 ms
  - p95 ITL: 17.3 ms
- Exact warm-cache saturation was not pursued because this already established a large locality effect.

### Analysis
- 18,712 input / 372 output tokens
- 0.7 RPS test:
  - achieved: 0.65 req/s
  - reported logical throughput: 12,433 tok/s
  - p95 TTFT: 1,367 ms
  - p95 ITL: 21.3 ms

### Main observation
Single-GPU cache locality materially changes capacity. Warm-prefix chat handled
far more logical traffic than cold chat while remaining within SLA. Therefore
a single-GPU warm-cache result cannot safely be multiplied by N for Beacon's
current round-robin deployment.

## Workload mix by token volume

Although 91% of requests are chat, chat represents only 72.7% of total tokens
because analysis requests are much larger.

- Chat: 72.7% of logical token volume
- Analysis: 27.3% of logical token volume

I use token-volume shares, not request-count shares, when estimating mixed-workload
GPU capacity.

In [17]:
chat_share = 0.727324
analysis_share = 0.272676

chat_cold = 15559
chat_warm = 46627
analysis_capacity = 12433

cache_hit_rate = 0.71

estimated_chat_capacity = (
    cache_hit_rate * chat_warm
    + (1 - cache_hit_rate) * chat_cold
)

estimated_chat_capacity

37617.28

In [18]:
mixed_capacity = 1 / (
    chat_share / estimated_chat_capacity
    + analysis_share / analysis_capacity
)

mixed_capacity

24232.748084139588

In [19]:
peak_tokens_per_sec = 1_000_000_000 / 3600

required_gpus = peak_tokens_per_sec / mixed_capacity

peak_tokens_per_sec, required_gpus

(277777.77777777775, 11.462908656223938)

## First-pass stock vLLM sizing

Assumption: Beacon's reported 71% cache hit rate can be approximated by
interpolating between my measured cold- and warm-prefix chat cases.

- Estimated chat capacity: 37,617 logical tok/s/GPU
- Estimated mixed-workload capacity: 24,233 logical tok/s/GPU
- Peak demand: 277,778 logical tok/s
- Stock vLLM requirement: 11.46 GPUs → 12 H100s

This is a sizing assumption, not a direct measurement. The 71% dashboard
metric is poorly defined, and linear interpolation between warm/cold capacity
is therefore the weakest step in this estimate.

In [20]:
claimed_improvement = 1.45

our_capacity = mixed_capacity * claimed_improvement
our_required_gpus = peak_tokens_per_sec / our_capacity

our_capacity, our_required_gpus

(35137.4847220024, 7.905454245671682)

In [21]:
our_gpus = 8
our_gpu_price = 3.60

our_hourly_cost = our_gpus * our_gpu_price

tessera_24_hourly = 24 * 2.40
tessera_16_hourly = 16 * 2.95
tessera_8_hourly = 8 * 3.40

our_hourly_cost, tessera_24_hourly, tessera_16_hourly, tessera_8_hourly

(28.8, 57.599999999999994, 47.2, 27.2)

In [22]:
avg_million_tokens_per_hour = 500

our_cost_per_million = our_hourly_cost / avg_million_tokens_per_hour
tessera_24_cost_per_million = tessera_24_hourly / avg_million_tokens_per_hour
tessera_16_cost_per_million = tessera_16_hourly / avg_million_tokens_per_hour
tessera_8_cost_per_million = tessera_8_hourly / avg_million_tokens_per_hour

(
    our_cost_per_million,
    tessera_24_cost_per_million,
    tessera_16_cost_per_million,
    tessera_8_cost_per_million,
)

(0.0576, 0.11519999999999998, 0.09440000000000001, 0.0544)

In [23]:
uplift_needed_for_8 = peak_tokens_per_sec / (8 * mixed_capacity)
uplift_needed_for_8

1.4328635820279922

## Quote decision

My first-pass model estimates that stock vLLM requires ~11.46 H100s at
Beacon's peak, so I round this to 12 GPUs. Tessera sells in 8-GPU node
increments, making 16 H100s ($47.20/hr) the smallest Tessera configuration
that clears that estimate.

Applying the claimed 1.45x tokens/GPU-hour improvement gives a theoretical
requirement of 7.91 GPUs. I would not quote 8 because it leaves essentially
no margin: the minimum uplift required for 8 GPUs is ~1.433x, only slightly
below the 1.45x claim.

Recommendation: quote 9x H100 at $3.60/GPU-hour = $32.40/hour.

At Beacon's 360B tokens/month (~500M tokens/hour average):
- Our quote: \$0.0648 per million tokens
- Tessera 16-GPU tier: \$0.0944 per million tokens
- Relative savings: ~31% per token

The ninth GPU is deliberate capacity margin against uncertainty in the
71% cache metric, workload sampling, and multi-GPU cache locality.

In [25]:
quote_gpus = 9
quote_hourly = quote_gpus * 3.60
quote_cost_per_million = quote_hourly / 500

quote_hourly, quote_cost_per_million

(32.4, 0.0648)

In [26]:
savings = 1 - 0.0648 / 0.0944
savings

0.31355932203389836